# ir_calendar_check_landing — 驗證 consume 後的 Delta 與 Volume

- 用途：`ir_calendar_consume_batches` 在 **`api_base_url` 留空**（資料只進 Databricks、沒送系統 API）跑完之後，確認 bronze / silver / Volume 三邊都對得上，再決定要不要補送 API。
- **只讀**：不寫表、不寫 Volume、不動游標，可以隨時重跑，也可以在 consume job 跑一半時看。
- 輸入：`b_{domain}_batch_log`、`b_{domain}_record`、`s_{domain}_{company, conference, summary, document_file}`、Volume `<volume_root>`
- 輸出：stdout 的 PASS / FAIL / INFO 清單（[c20] 彙總），不產生任何表
- 參數（widgets）：`catalog`、`schema`、`domain`、`volume_root`、`batch_id`、`sample_rows`、`file_check_limit`、`verify_sha256`
- 排程：不排程，人工執行；job cluster 或 all-purpose 都可以
- 負責人 / 更新日期：（填）/ 2026-09-21

## 檢查了什麼

| cell | 檢查 |
|---|---|
| [c10] `check_batch_log` | 最近 10 批的處理結果；決定檢查對象 `target`（widget 留空 = 最新一批）。確認 `status = SUCCESS`、`landed + archived = files_total`、`api_results` 是空的（= 確實沒送系統 API） |
| [c11] `check_cursor` | Volume 上的 `_cursor.json` 是否等於 batch_log 最新成功批次的 `seq` |
| [c12] `check_bronze` | 這一批各 `record_type` 的列數；bronze 列數 = `batch_log.bronze_rows` = `files_total + 1`（manifest 自己那列）；`payload` 無空值、實體檔列都有 `volume_path`、`batch_path` 不重複 |
| [c13] `check_silver` | 四張 silver 的 MERGE 鍵唯一、鍵無 NULL、`updated_at` 有值、本批有進來；另檢 `conference.conference_date` / `stock_code` 無 NULL、`document_file.volume_path` 都在 `volume_root` 底下 |
| [c14] `sample_silver` | 每張表本批 `limit sample_rows` 列，肉眼看欄位 |
| [c15] `check_volume_archive` | `<volume_root>/ir_calendar/batches/<批次>/` 有 `manifest.json`、檔數 = `archived_files + 1`、沒有殘留 `.tmp` |
| [c16] `check_volume_files` | 從 `s_*_document_file` 取 `volume_path`（上限 `file_check_limit`）確認檔案存在、大小 = manifest `bytes`；`verify_sha256 = true` 時重算雜湊。另印分類目錄長相，並確認沒有檔案掉進 `uncategorized` / `_unknown_company` |

## 全過之後要補送系統 API

1. 編輯 `<volume_root>/ir_calendar/_cursor.json`，把 `last_seq` 改回要重送的批次之前（第一次上線就是 `0`）
2. `ir_calendar_consume_batches` 填 `api_base_url`（到 `/api/v1` 為止，不含結尾斜線）與 `api_key_secret`（`scope/key`，值放 `dbutils.secrets`）
3. 重跑。重跑是冪等的：Volume 同名覆蓋、bronze `DELETE WHERE batch_id` 後 append、silver MERGE、API 帶 `Idempotency-Key`

Cell 標籤規則見 `docs/conventions.md`。


In [ ]:
# [c01] params
# 只讀：這份 notebook 不寫表、不寫 Volume、不動游標，可以隨時重跑。
# catalog / schema / volume_root 必須跟 ir_calendar_consume_batches 那次執行用的一樣，否則查到別的地方。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")
dbutils.widgets.text("batch_id", "")             # 空 = 用 batch_log 最新那一批
dbutils.widgets.text("sample_rows", "5")         # 每張表抽樣列數（一律走 limit，不拉全表）
dbutils.widgets.text("file_check_limit", "200")  # 最多實際檢查幾個落地檔
dbutils.widgets.text("verify_sha256", "false")   # true：重算落地檔 sha256 與 silver 對照（要讀檔，慢）

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "batch_id": dbutils.widgets.get("batch_id").strip(),
    "sample_rows": int(dbutils.widgets.get("sample_rows")),
    "file_check_limit": int(dbutils.widgets.get("file_check_limit")),
    "verify_sha256": dbutils.widgets.get("verify_sha256").strip().lower() == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["domain"], "catalog / schema / domain 不可為空"
assert cfg["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
print(cfg)


In [ ]:
# [c02] imports
# 常數與 ir_calendar_consume_batches 的 [c02] / [c04] 對齊：那邊改了 CATEGORY_DIR、LAYER_PREFIX、TABLE_KEYS，這裡要一起改。
import hashlib
import json
import os

from pyspark.sql import functions as F

LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}
CONTROL_DIR = "ir_calendar"                                            # <volume_root>/ir_calendar/：游標與批次歸檔
CATEGORY_DIRS = ["panel_peer", "brand_customer", "supplier", "uncategorized"]   # consume [c02] CATEGORY_DIR 的值域

SILVER_SHORTS = ["company", "conference", "summary", "document_file"]  # 順序同 consume 的 SILVER_TRANSFORMS
SILVER_KEYS = {                                                        # = consume [c04] TABLE_KEYS
    "company": ["company_key"],
    "conference": ["company_key", "period", "revision"],
    "summary": ["company_key", "period"],
    "document_file": ["volume_path"],
}


In [ ]:
# [c03] check_helpers
# 檢查結果收集器。每項檢查只記錄不 raise，讓全部跑完再由 [c20] 一次看；真正擋不下去的才用 assert。
CHECKS: list[tuple[str, bool | None, str]] = []


def check(name: str, ok: bool, detail: str = "") -> bool:
    CHECKS.append((name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f" — {detail}" if detail else ""))
    return bool(ok)


def note(name: str, detail: str) -> None:
    """不判定對錯，只記錄觀察值。"""
    CHECKS.append((name, None, detail))
    print(f"  INFO  {name} — {detail}")


def table_name(layer: str, short: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{LAYER_PREFIX[layer]}{cfg['domain']}_{short}"


def null_key_cond(keys: list[str]):
    cond = F.lit(False)
    for k in keys:
        cond = cond | F.col(k).isNull()
    return cond


In [ ]:
# [c10] check_batch_log
# 從 job 自己的紀錄開始：這批跑完沒有、有沒有東西送出去。決定後面所有 cell 的檢查對象 target。
log_table = table_name("bronze", "batch_log")
print(f"[{log_table}] 最近 10 批")
recent = spark.table(log_table).orderBy(F.col("seq").desc()).limit(10).collect()   # 小表 + limit
for r in recent:
    print(f"  seq={r.seq} {r.batch_id}  {r.status:<8} kind={r.kind} files={r.files_total} "
          f"landed={r.landed_files} archived={r.archived_files} bronze={r.bronze_rows} "
          f"api={len(r.api_results or [])} processed_at={r.processed_at}")

assert recent, f"{log_table} 是空的：consume job 還沒成功跑過，先看那支 job 的 stdout"
target = cfg["batch_id"] or recent[0].batch_id
row = next((r for r in recent if r.batch_id == target), None)
assert row is not None, f"batch_id={target} 不在最近 10 批裡；確認拼字，或直接留空用最新一批"
print(f"\n檢查對象：{target}（seq={row.seq}，kind={row.kind}）\n")

check(f"{target} status = SUCCESS", row.status == "SUCCESS", row.error or "")
check("landed + archived = files_total",
      (row.landed_files or 0) + (row.archived_files or 0) == (row.files_total or 0),
      f"landed={row.landed_files} archived={row.archived_files} files_total={row.files_total}")
check("尚未轉送系統 API（api_base_url 留空時的預期結果）", not (row.api_results or []),
      f"api_results={len(row.api_results or [])} 筆" if row.api_results else "")
failed = [r.batch_id for r in recent if r.status != "SUCCESS"]
check("最近 10 批沒有 FAILED", not failed, ", ".join(failed))
note("manifest counts", str(dict(row.counts or {})))


In [ ]:
# [c11] check_cursor
# 游標存在 Volume 不在 Delta。它應該等於 batch_log 裡最新一筆 SUCCESS 的 seq；對不上代表有批次寫了表卻沒推游標（或反過來）。
cursor_file = f"{cfg['volume_root']}/{CONTROL_DIR}/_cursor.json"
if os.path.exists(cursor_file):
    with open(cursor_file, encoding="utf-8") as f:
        cursor = json.load(f)
    print(f"{cursor_file}\n  {cursor}\n")
    ok_seq = max([r.seq for r in recent if r.status == "SUCCESS"], default=0)
    check("游標 = 最新成功批次 seq", int(cursor.get("last_seq") or 0) == ok_seq,
          f"cursor={cursor.get('last_seq')} batch_log={ok_seq}")
    note("補送 API 要改的值", f"把 last_seq 改回 {int(cursor.get('last_seq') or 0) - 1} 之前的值，這些批才會重跑")
else:
    check("游標檔存在", False, f"找不到 {cursor_file}")


In [ ]:
# [c12] check_bronze
# bronze 是原樣落地，一檔一列 + manifest 自己一列。只看這一批（batch_id 過濾放最前面），不掃全表。
rec_table = table_name("bronze", "record")
b = spark.table(rec_table).filter(F.col("batch_id") == F.lit(target)).cache()   # 同一份要跑多個 count
by_type = {r["record_type"]: r["n"]
           for r in b.groupBy("record_type").agg(F.count("*").alias("n")).collect()}
total = sum(by_type.values())
print(f"[{rec_table}] batch_id={target}")
for k, v in sorted(by_type.items()):
    print(f"  {k:<20} {v}")
print(f"  {'合計':<18} {total}\n")

check("bronze 列數 = batch_log.bronze_rows", total == (row.bronze_rows or 0),
      f"bronze={total} batch_log={row.bronze_rows}")
check("bronze 列數 = files_total + 1（manifest 那列）", total == (row.files_total or 0) + 1,
      f"實際 {total}，files_total={row.files_total}")
check("manifest 自己有一列", by_type.get("batch_manifest", 0) == 1,
      f"batch_manifest={by_type.get('batch_manifest', 0)}")
check("payload 沒有空值", b.filter(F.col("payload").isNull() | (F.length("payload") == 0)).count() == 0)
n_no_path = b.filter((F.col("record_type") == "ir_document_file")
                     & F.col("volume_path").isNull()).count()
check("實體檔列都有 volume_path", n_no_path == 0, f"{n_no_path} 列缺路徑")
n_dup = b.groupBy("batch_path").agg(F.count("*").alias("n")).filter("n > 1").count()
check("同一批沒有重複 batch_path", n_dup == 0, f"{n_dup} 個重複（重跑沒清乾淨的徵兆）")
b.unpersist()


In [ ]:
# [c13] check_silver
# 四張 silver：MERGE 鍵唯一、鍵沒有 NULL、本批有進來。silver 都是小表，count 可接受。
for short in SILVER_SHORTS:
    t = table_name("silver", short)
    keys = SILVER_KEYS[short]
    if not spark.catalog.tableExists(t):
        check(f"{short}: 表存在", False, f"{t} 不存在，先跑 ir_calendar_init_tables")
        continue
    df = spark.table(t)
    n_all = df.count()
    n_batch = df.filter(F.col("batch_id") == F.lit(target)).count()
    n_dup = df.groupBy(*keys).agg(F.count("*").alias("n")).filter("n > 1").count()
    n_nullkey = df.filter(null_key_cond(keys)).count()
    print(f"[{t}] 全表 {n_all} 列，本批 {n_batch} 列，鍵 {keys}")
    check(f"{short}: MERGE 鍵唯一", n_dup == 0, f"{n_dup} 組重複")
    check(f"{short}: 鍵沒有 NULL", n_nullkey == 0, f"{n_nullkey} 列")
    check(f"{short}: updated_at 都有值", df.filter(F.col("updated_at").isNull()).count() == 0)
    note(f"{short}: 列數", f"全表 {n_all}，本批 {n_batch}")

# 各表的重點欄位（爬蟲缺值或 ANSI 解析失敗會在這裡現形）
conf_t = table_name("silver", "conference")
if spark.catalog.tableExists(conf_t):
    conf = spark.table(conf_t)
    check("conference: conference_date 沒有 NULL", conf.filter(F.col("conference_date").isNull()).count() == 0,
          "日期是台北曆日，爬蟲端已換算；NULL 代表來源沒給或格式不是 yyyy-MM-dd")
    check("conference: stock_code 沒有 NULL", conf.filter(F.col("stock_code").isNull()).count() == 0,
          "送系統 API 時 stock_code 是必填，查無公司該列會失敗")
doc_t = table_name("silver", "document_file")
if spark.catalog.tableExists(doc_t):
    bad = spark.table(doc_t).filter(~F.col("volume_path").startswith(cfg["volume_root"])).count()
    check("document_file: volume_path 都在 volume_root 底下", bad == 0, f"{bad} 列指到別的地方")


In [ ]:
# [c14] sample_silver
# 開發用抽樣：每張表本批 limit 幾列，肉眼看欄位有沒有跑掉。display() 只在這個 cell 用，不要拿掉 limit。
for short in SILVER_SHORTS:
    t = table_name("silver", short)
    if not spark.catalog.tableExists(t):
        continue
    print(f"[{t}] 本批前 {cfg['sample_rows']} 列")
    display(spark.table(t).filter(F.col("batch_id") == F.lit(target)).limit(cfg["sample_rows"]))


In [ ]:
# [c15] check_volume_archive
# Volume 控制區：<volume_root>/ir_calendar/batches/<批次>/ 放這批的 JSON 原檔與 manifest.json。
archive_dir = f"{cfg['volume_root']}/{CONTROL_DIR}/batches/{target}"
if not os.path.isdir(archive_dir):
    check("歸檔目錄存在", False, f"找不到 {archive_dir}")
else:
    top = sorted(os.listdir(archive_dir))
    n_files = 0
    print(f"{archive_dir}")
    for name in top:
        p = f"{archive_dir}/{name}"
        if os.path.isdir(p):
            inner = sorted(os.listdir(p))
            n_files += len(inner)
            print(f"  {name}/  {len(inner)} 個，前 5：{inner[:5]}")
        else:
            n_files += 1
            print(f"  {name}  {os.path.getsize(p)} bytes")
    check("歸檔目錄有 manifest.json", "manifest.json" in top)
    check("歸檔檔數 = archived_files + 1（manifest）", n_files == (row.archived_files or 0) + 1,
          f"實際 {n_files}，archived_files={row.archived_files}")
    check("沒有殘留的 .tmp", not [n for n in top if n.endswith(".tmp")], "寫入中斷的徵兆")


In [ ]:
# [c16] check_volume_files
# 實體檔：拿 silver 的 volume_path 去確認檔案真的在、大小對得上。抽樣上限 file_check_limit，不掃整個 Volume。
doc_t = table_name("silver", "document_file")
docs = (spark.table(doc_t).filter(F.col("batch_id") == F.lit(target))
        .select("volume_path", "bytes", "sha256", "file_name")
        .limit(cfg["file_check_limit"]).collect())
print(f"檢查 {len(docs)} 個落地檔（上限 {cfg['file_check_limit']}，本批 landed={row.landed_files}）")

missing, wrong_size, wrong_hash = [], [], []
for d in docs:
    if not d.volume_path or not os.path.exists(d.volume_path):
        missing.append(d.file_name)
        continue
    if d.bytes is not None and os.path.getsize(d.volume_path) != d.bytes:
        wrong_size.append(d.file_name)
    if cfg["verify_sha256"] and d.sha256:
        h = hashlib.sha256()
        with open(d.volume_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        if h.hexdigest() != d.sha256:
            wrong_hash.append(d.file_name)

check("落地檔都存在", not missing, ", ".join(missing[:5]))
check("檔案大小 = manifest bytes", not wrong_size, ", ".join(wrong_size[:5]))
if cfg["verify_sha256"]:
    check("sha256 相符", not wrong_hash, ", ".join(wrong_hash[:5]))
else:
    note("sha256", "未驗（verify_sha256 = false）")

# 目錄長相：<volume_root>/<分類>/<公司代稱>/<檔名> 兩層
print("\n分類目錄：")
for cat in CATEGORY_DIRS:
    p = f"{cfg['volume_root']}/{cat}"
    if os.path.isdir(p):
        slugs = sorted(os.listdir(p))
        print(f"  {cat}/  {len(slugs)} 家公司，前 5：{slugs[:5]}")
    else:
        print(f"  {cat}/  (無)")
unc = f"{cfg['volume_root']}/uncategorized"
check("沒有檔案掉進 uncategorized", not os.path.isdir(unc) or not os.listdir(unc),
      "有的話代表 manifest 的 category 缺值")
unk = [f"{cfg['volume_root']}/{c}/_unknown_company" for c in CATEGORY_DIRS]
check("沒有檔案掉進 _unknown_company", not [p for p in unk if os.path.isdir(p) and os.listdir(p)],
      "有的話代表 manifest 的 company_slug 缺值")


In [ ]:
# [c20] summary
# 一次看完。FAIL 的項目要先解決，再去填 api_base_url 把資料送進系統。
n_pass = sum(1 for _, ok, _ in CHECKS if ok is True)
n_fail = sum(1 for _, ok, _ in CHECKS if ok is False)
print(f"批次 {target}：檢查 {n_pass + n_fail} 項，PASS {n_pass} / FAIL {n_fail}\n")
for name, ok, detail in CHECKS:
    mark = {True: "PASS", False: "FAIL", None: "INFO"}[ok]
    print(f"{mark}  {name}" + (f" — {detail}" if detail else ""))

if n_fail:
    print(f"\n有 {n_fail} 項沒過，先處理完再送 API。")
else:
    print("\n全部通過。要補送系統 API：")
    print(f"  1. 編輯 {cfg['volume_root']}/{CONTROL_DIR}/_cursor.json，把 last_seq 改回要重送的批次之前")
    print("  2. ir_calendar_consume_batches 填 api_base_url（到 /api/v1 為止）與 api_key_secret（scope/key）")
    print("  3. 重跑；重跑是冪等的（Volume 覆蓋、bronze 刪後重寫、silver MERGE、API 有 Idempotency-Key）")
